In [ ]:
# accuracy: 0.8044692737430168
# precision: 0.7746478873239436
# recall: 0.7432432432432432

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [4]:
df = pd.read_csv('titanic_train.csv')
df['Sex'] = df['Sex'].map({'male':0, 'female':1})
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Age'] = df['Age'].fillna(df.groupby(['Pclass', 'Sex'])['Age'].transform('median'))
df['HasCabin'] = df['Cabin'].apply(lambda x: 0 if pd.isna(x) else 1)
df.drop('Cabin', axis=1, inplace=True)
df["Title"] = df["Name"].str.extract("([A-Za-z]+)\.", expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col',\
 	'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Embarked'] = df['Embarked'].map({'S':1, 'Q':2, 'C':3})
df['Title'] = df['Title'].map({'Mr':1, 'Mrs':2,  'Miss':3, 'Master':4, 'Rare':5})
# print(df.info())
# print(df.nunique())
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked', 'HasCabin', 'Title'],
      dtype='object')

In [58]:
X = df.drop(columns=['PassengerId','Name','Survived','Ticket'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
param_grid = ({'knn__n_neighbors': [3,5,7,9,11,13,15,17]})

classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='recall'
)


classifierCV.fit(X_train, y_train)
y_pred = classifierCV.predict(X_test)


In [59]:
print(f"accuracy: {accuracy_score(y_test, y_pred)}")
print(f"precision: {precision_score(y_test, y_pred)}")
print(f"recall: {recall_score(y_test, y_pred)}")
print(classifierCV.best_params_)

accuracy: 0.8268156424581006
precision: 0.8028169014084507
recall: 0.7702702702702703
{'knn__n_neighbors': 15}
